<a href="https://colab.research.google.com/github/ajaynice1996/DIY_MRI_Workshop_II_Recon/blob/main/Day_3_Notebook_1_3_zssr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive
Cloning into 'DIY_MRI_Workshop_II_Recon'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 168 (delta 26), reused 7 (delta 1), pack-reused 109 (from 2)
Receiving objects: 100% (168/168), 111.76 MiB | 15.49 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [32]:
!git clone https://github.com/ajaynice1996/DIY_MRI_Workshop_II_Recon

fatal: destination path 'DIY_MRI_Workshop_II_Recon' already exists and is not an empty directory.


In [34]:
pwd

'/content'

In [8]:
!pip install nilearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
cud

In [44]:
!pip install colorama
!pip -q install roipoly

  Preparing metadata (setup.py) ... done


In [38]:
import sys

REPO_ROOT = "/content/DIY_MRI_Workshop_II_Recon"

sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, f"{REPO_ROOT}/zssr")

In [39]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from typing import Dict, Tuple

# SRR_SS_Mon
from SRR_SS_Mon.data_read import PairedMRI

# ZSSR
from zssr.ZSSR_master import ZSSR
from zssr.ZSSR_master import configs, configs_2

# Utilities
from zssr.utils import compute_aes

In [41]:
from nilearn import plotting
from nibabel.viewers import OrthoSlicer3D
import tensorflow as tf
# import pydicom # Commented out as it was previously for potential import errors
import numpy as np
import matplotlib
# matplotlib.use('TkAgg')  # or 'Qt5Agg' depending on what's installed
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
# from pydicom.filereader import dcmread # Commented out as it was previously for potential import errors
from tensorflow.keras import backend as K
import scipy.io as sio

# Clear the current TensorFlow/Keras session
K.clear_session()

print(tf.config.list_physical_devices('GPU'))  # Check if GPU is visible
print(tf.config.list_physical_devices('CPU'))  # Check if CPU is visible

if tf.config.list_physical_devices('GPU'):
    print("CUDNN detected!")
else:
    print("Display the data using OrthoSlicer3D")
    # Removed img.dataobj as img is not defined here yet
    # OrthoSlicer3D(img.dataobj).show()
    # plotting.plot_anat(img, title="3D TSC Image")
    plt.show()

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
CUDNN detected!


In [46]:
import warnings
# Suppress DeprecationWarnings from jupyter_client
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
print("DeprecationWarnings from jupyter_client are now suppressed.")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

DeprecationWarnings from jupyter_client are now suppressed.


In [47]:
# Note: Undo CUDNN detected!")

from do_zssr_collage import *
from nifti_write import make_nifti
# from LF_simulation_functions import read_nifti
from colorama import Fore, Back, Style
import itertools
from scipy.ndimage import sobel

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

viewing = False
ds_to_process = 4
target_resolution_fact = [1, 1, 2]
scale_factor = target_resolution_fact[2]  # Z-axis scaling factor
snr_component = False

max_iters = 10
min_iters = 5
# Define parameter options
widths = [32] # width of the filters in the conv layers
depths = [4] # depth of the network
crop_sizes = [32] # size of the patches to crop from the image
noise_stds = [0.0] # standard deviation of the noise to add to the image

# Load dataset
training_path = "/content/DIY_MRI_Workshop_II_Recon/data/Training_data"
dataset = PairedMRI(training_path)
kernel_path = '/Users/sairamgeethanath/Documents/Contributions/Tools/Projects/R21/lf-brain-tracking/src/ZSSR_master/kernel_example/BSD100_100_lr_rand_ker_c_X2_0.mat'

kernel_files = ['%s_%d.mat' % (kernel_path[:-4], ind) for ind in range(len([1, 2]))]
# List subjects
print(dataset.subjects)

for i, subject_id in enumerate(dataset.subjects[0:1]):  # take first 5 subjects
    print(Fore.CYAN + f"\n=== Processing Subject {i+1}: {subject_id} ===" + Style.RESET_ALL)

    # Get LF & HF images
    subject_LF_Monash = dataset.get_subject_image(subject_id, "T1", 'LF', visible=False, target_spacing=
                                           [1, 1, 1])
    subject_LF_ZSSR = dataset.get_subject_image(subject_id, "T1", 'LF', visible=False,
                                           target_spacing=target_resolution_fact)
    subject_HF = dataset.get_subject_image(subject_id, "T1", 'HF', visible=False,target_spacing=
                                           [1, 1, 1])

    # Convert to uint8 NIfTI
    subject_LF_Monash = nib.Nifti1Image(subject_LF_Monash.get_fdata().astype(np.uint8),
                                 subject_LF_Monash.affine, subject_LF_Monash.header)
    subject_LF_ZSSR = nib.Nifti1Image(subject_LF_ZSSR.get_fdata().astype(np.uint8),
                                 subject_LF_ZSSR.affine, subject_LF_ZSSR.header)
    subject_HF = nib.Nifti1Image(subject_HF.get_fdata().astype(np.uint8),
                                 subject_HF.affine, subject_HF.header)

    img_data = subject_LF_ZSSR.get_fdata()

    if viewing:
        print("Displaying LF ZSSR image data using OrthoSlicer3D")
        OrthoSlicer3D(img_data).show()

    print("Shape of img_data:", img_data.shape)
    # Generate unique NIfTI filename per subject

    nifti_file = f"Data/{subject_id}_T1.nii.gz"

    hdr = subject_LF_ZSSR.header
    pixdim = hdr['pixdim']

    # Print info
    print(Fore.GREEN + 'PROCESSING NIFTI FILE METADATA' + Style.RESET_ALL)
    print("Dimensions:", hdr.get_data_shape())
    print("Voxel Sizes:", hdr.get_zooms())
    print("Data Type:", hdr.get_data_dtype())
    print("Intent:", hdr.get_intent())

    # Get the HF data for comparison
    subject_HF_data = subject_HF.get_fdata()
    subject_HF_data = subject_HF_data / np.max(subject_HF_data)
    subject_HF_data = (subject_HF_data * 4095).astype(np.uint16)

    subject_LF_Monash_data = subject_LF_Monash.get_fdata()
    subject_LF_Monash_data = subject_LF_Monash_data / np.max(subject_LF_Monash_data)
    subject_LF_Monash_data = (subject_LF_Monash_data * 4095).astype(np.uint16)

    # start a clock so that we can compute the time taken for all parameter combinations
    import time
    start_time = time.time()
    # Create all combinations
    param_combinations = list(itertools.product(widths, depths, crop_sizes, noise_stds))
    # print(param_combinations)
    for idx, (width, depth, crop_size, noise_std) in enumerate(param_combinations):
        # Print current combination
        print(Fore.YELLOW + f"Running ZSSR with width={width}, depth={depth}, crop_size={crop_size}, noise_std={noise_std}, idx = {idx}" + Style.RESET_ALL)

        # change of recon.config
        recon_config = configs.Config(width=width, depth=depth, crop_size=crop_size, noise_std=noise_std)
        # recon_config.scale_factors = [[np.sqrt(target_resolution_fact[0]), 1]]
        recon_config.scale_factors = [[(target_resolution_fact[0]), 1]]
        recon_config.max_iters = max_iters
        recon_config.min_iters = min_iters
        recon_config.width = width
        recon_config.depth = depth
        recon_config.noise_std = noise_std
        recon_config.crop_size = crop_size
        num_rows = 16
        num_cols = 14
        print('Interpolation method:', recon_config.upscale_method)
        # Run ZSSR

        print('Passing through ZSSR ..........')

        im_lf_sim_zssr = do_ZSSR_steps(
            img=img_data, recon_conf=recon_config, num_cols=num_cols, num_rows=num_rows,
            fname_zssr=nifti_file, fspec='', scale_fact=scale_factor, dims = 1, ground_truth=None, kernel=None)

        # Compute PSNR/SSIM/AES between im_lf_sim_zssr and subject_HF
        # (Assuming subject_HF is already loaded as a NIfTI image)

        # Ensure all images are in the same dynamic range 0 - 1
        im_lf_sim_zssr_yz = im_lf_sim_zssr / np.max(im_lf_sim_zssr)

        print(Fore.GREEN + "Shape of im_lf_sim_zssr_yz:" + str(im_lf_sim_zssr_yz.shape) + Style.RESET_ALL)

        # Now let us switch the two axes to also perform ZSSR in the other plane
        img_data_xz = np.swapaxes(img_data, 0, 1)
        print(Fore.GREEN + "Shape of img_data_xz:" + str(img_data_xz.shape) + Style.RESET_ALL)

        # Run ZSSR on the swapped axes
        im_lf_sim_zssr_xz = do_ZSSR_steps(
            img=img_data_xz, recon_conf=recon_config, num_cols=num_cols, num_rows=num_rows,
            fname_zssr=nifti_file, fspec='', scale_fact=scale_factor, dims=1, ground_truth=None, kernel=None)

        # Swap axes back to original orientation
        im_lf_sim_zssr_xz = np.swapaxes(im_lf_sim_zssr_xz, 0, 1)

        # Combine the two ZSSR results
        # Combine the two ZSSR results by selecting, for each voxel, the value from the volume (yz or xz)
        # that has the higher local gradient magnitude (i.e., sharper neighborhood)

        # Compute gradient magnitude for both volumes
        grad_yz = np.sqrt(
            sobel(im_lf_sim_zssr_yz, axis=0, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_yz, axis=1, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_yz, axis=2, mode='reflect')**2
        )
        grad_xz = np.sqrt(
            sobel(im_lf_sim_zssr_xz, axis=0, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_xz, axis=1, mode='reflect')**2 +
            sobel(im_lf_sim_zssr_xz, axis=2, mode='reflect')**2
        )

        # For each voxel, pick the value from the sharper (higher gradient) volume
        mask = grad_yz >= grad_xz
        im_lf_sim_zssr_combined = np.where(mask, im_lf_sim_zssr_yz, im_lf_sim_zssr_xz)

        # Make sure all comparisons are between 0 to 4095 to match 12 bit DICOM range
        im_lf_sim_zssr_combined = (im_lf_sim_zssr_combined * 4095).astype(np.uint16)
        im_lf_sim_zssr = im_lf_sim_zssr_combined

        # Compute PSNR
        psnr_value_monash = psnr(subject_LF_Monash_data, subject_HF_data)
        psnr_value_zssr = psnr(im_lf_sim_zssr, subject_HF_data)

        # Compute SSIM
        ssim_value_monash = ssim(subject_LF_Monash_data, subject_HF_data, data_range=4095)
        ssim_value_zssr = ssim(im_lf_sim_zssr, subject_HF_data, data_range=4095)

        # Compute AES
        aes_value_HF = compute_aes(subject_HF_data)
        aes_value_monash = compute_aes(subject_LF_Monash_data)
        aes_value_zssr = compute_aes(im_lf_sim_zssr)

        # Print PSNR, SSIM, and AES values in a table format
        print(Fore.GREEN + f"{'Method':<15}{'PSNR':<15}{'SSIM':<15}{'AES':<15}" + Style.RESET_ALL)
        print(Fore.GREEN + f"{'Monash LF':<15}{psnr_value_monash:<15.4f}{ssim_value_monash:<15.4f}{aes_value_monash:<15.4f}" + Style.RESET_ALL)
        print(Fore.GREEN + f"{'ZSSR':<15}{psnr_value_zssr:<15.4f}{ssim_value_zssr:<15.4f}{aes_value_zssr:<15.4f}" + Style.RESET_ALL)
        print(Fore.GREEN + f"{'HF (Ground Truth)':<15}{'N/A':<15}{'N/A':<15}{aes_value_HF:<15.4f}" + Style.RESET_ALL)
        # # Save output with subject-specific name and config values
        zssr_fname = (
            f"./Data/Results_ss/{subject_id}_T1_zssr_w{width}_d{depth}_c{crop_size}_n{noise_std}_test1{snr_component}.nii.gz"
        )

        # make_nifti(im_lf_sim_zssr, fname=zssr_fname, mask=False,
        #            res=[pixdim[1], pixdim[2], pixdim[3]], dim_info=[0, 1, 2])

        # print(Fore.YELLOW + f"Saved ZSSR output -> {zssr_fname}" + Style.RESET_ALL)
        viewing = False
        if viewing:
            # Display a panel of the mid coronal slice for HF, Monash LF, LF input, and LF ZSSR
            mid_slice = subject_HF_data.shape[1] // 2

            fig, axes = plt.subplots(1, 4, figsize=(16, 4))
            axes[0].imshow(np.rot90(subject_HF_data[:, mid_slice, :], k=1), cmap='gray')
            axes[0].set_title('High Field (HF)')
            axes[0].axis('off')

            axes[1].imshow(np.rot90(img_data[:, mid_slice, :], k=1), cmap='gray')
            axes[1].set_title('LF Input')
            axes[1].axis('off')

            axes[2].imshow(np.rot90(subject_LF_Monash_data[:, mid_slice, :], k=1), cmap='gray')
            axes[2].set_title('Monash LF')
            axes[2].axis('off')

            axes[3].imshow(np.rot90(im_lf_sim_zssr[:, mid_slice, :], k=1), cmap='gray')
            axes[3].set_title('LF ZSSR')
            axes[3].axis('off')

            plt.tight_layout()
            plt.show()

        if viewing:
            OrthoSlicer3D(im_lf_sim_zssr).show()
            plt.show()
    end_time = time.time()
    total_time = end_time - start_time
    print(Fore.CYAN + f"Total time for all parameter combinations: {total_time:.2f} seconds" + Style.RESET_ALL)

Num GPUs Available:  1
['POCEMR001', 'POCEMR003', 'POCEMR004', 'Phantom_2.nii.gz', 'phantom_1.nii.gz']

=== Processing Subject 1: POCEMR001 ===
Shape of img_data: (224, 224, 80)
PROCESSING NIFTI FILE METADATA
Dimensions: (224, 224, 80)
Voxel Sizes: (np.float32(1.0), np.float32(1.0), np.float32(2.0))
Data Type: uint8
Intent: ('none', (), '')
Running ZSSR with width=32, depth=4, crop_size=32, noise_std=0.0, idx = 0
Interpolation method: lanczos3
Passing through ZSSR ..........
(3584, 1120)
Num GPUs Available:  1
** Start training for sf= [1, np.float64(1.4142135623730951)]  **
sf: [1.         1.41421356] , iteration:  0 , loss:  70.25021
iteration:  0 reconstruct mse: 0.00023284883889631593 , true mse: None
** Done training for sf= [1, np.float64(1.4142135623730951)]  **
<class 'slice'>

Num GPUs Available:  1
** Start training for sf= [1, np.float64(1.4142135623730951)]  **
sf: [1.         1.41421356] , iteration:  0 , loss:  0.0
iteration:  0 reconstruct mse: 3.166702861842255e-05 , tr

KeyboardInterrupt: 